# Healthcare Example: Record-Level ML Lineage with DVC and MLflow on Amazon SageMaker AI

This notebook builds on the [foundational dataset-level lineage pattern](../foundational/), which links every model to the exact dataset version it was trained on via DVC. Dataset-level lineage tells you *which dataset* trained a model — but not which individual records are inside it. To answer "was record X in this model's training data?", you'd need to reconstruct the full dataset and search through it.

**Record-level lineage** closes that gap by adding a **manifest** — a structured index listing every record in each dataset version. The manifest is logged as an MLflow artifact on every training run, making individual records queryable directly from MLflow without pulling the full dataset from DVC. You'll learn how to:

- Version processed datasets with DVC and store them in Amazon S3
- Include a **record-level manifest** linking individual data records to dataset versions
- Track experiments and link models to specific data versions with MLflow
- Handle **record opt-out requests** with full audit trails
- **Query lineage**: "Which models were trained on record X's data?"

This pattern applies wherever you need to trace individual records through the ML lifecycle — healthcare, finance, or any domain where you must answer "which records trained this model?" and handle exclusion requests.

### Why DVC?

A manifest alone (listing which records to include) isn't enough — it doesn't capture what happened to the data during preprocessing (resizing, normalization, train/val splitting, augmentation). Two runs with the same manifest can produce different training data if preprocessing changes. **DVC versions the processed, ready-to-train dataset**, so `dvc pull` gives you the exact data that trained a given model — no reprocessing needed.

### MLflow on Amazon SageMaker AI

[MLflow on Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) makes it easier to track experiments and monitor performance of models and AI applications using a single tool. SageMaker AI provides a fully managed MLflow App that integrates natively with SageMaker AI Training jobs. Every training run stores the DVC commit hash, creating a complete chain: **Model → DVC commit → processed data + manifest → record IDs**.

---

### Demo Scenario: Healthcare Patient Opt-Out

This demo uses the [Montgomery County CXR Dataset](https://lhncbc.nlm.nih.gov/LHC-downloads/downloads.html#702702-tuberculosis-chest-x-ray-image-data-sets) — 138 chest X-rays across 2 diagnostic classes (normal, tuberculosis) from the National Library of Medicine — to demonstrate the compliance workflow:

1. **Upload & Register**: Upload raw chest X-ray images to S3, generate a patient manifest with consent tracking
2. **v1.0**: Process dataset for all consented patients, train model
3. **Opt-out**: A patient requests to opt out of model training — exclude their scans, create new dataset version
4. **v2.0**: Retrain model on clean dataset (without the opted-out patient)
5. **Audit**: Query which models used the patient's data, verify exclusion after opt-out date

## Prerequisites
---

This notebook can run in Amazon SageMaker Studio, a SageMaker AI notebook instance, or locally with AWS credentials configured.

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 3.7.0</strong> and the <strong>SageMaker Python SDK version 3.4.0</strong> and <strong>Python version 3.12</strong>
</div>

In [ ]:
# Install dependencies, clean install of SageMaker
# Dependency resolver warnings during install are expected and can be safely ignored
%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
%pip install --no-cache-dir -r ../requirements.txt

### Restart Your Kernel

In [ ]:
import sagemaker
import mlflow
import torch
from importlib.metadata import version

print(f"SageMaker SDK version: {version('sagemaker')}")
print(f"MLflow version: {mlflow.__version__}")
print(f"PyTorch version: {torch.__version__}")

## Part 1: Configure DVC for Data Versioning
---

We create a subdirectory with a git repository to store DVC metadata. The actual data is stored in Amazon S3.

**Note:** This example uses AWS CodeCommit, but DVC works with any Git provider (GitHub, GitLab, Bitbucket, etc.). Simply replace the `git remote add origin` URL with your repository URL and configure appropriate credentials. The key requirement is that your SageMaker AI execution role (or notebook IAM role) must have permissions to access the Git repository — for CodeCommit, this means `codecommit:GitPull` and `codecommit:GitPush` permissions.

In [ ]:
# Define the DVC repository name
dvc_repo_name = "cxr-dvc-demo"

In [ ]:
%%bash -s "$dvc_repo_name"

repo_name="$1"

# Create CodeCommit repository
aws codecommit create-repository --repository-name ${repo_name} \
    --repository-description "Chest X-ray classification with DVC versioning"

account=$(aws sts get-caller-identity --query Account --output text)
region=$(python -c "import boto3;print(boto3.Session().region_name)")
region=${region:-us-east-1}

mkdir -p ${repo_name}
cd ${repo_name}

# Initialize git repo
git init
git branch -M main  
git remote add origin codecommit::${region}://${repo_name}

# Configure git
git config --global user.email "user@domain.com"
git config --global user.name "user"
git config --global credential.helper '!aws codecommit credential-helper $@'
git config --global credential.UseHttpPath true

# Initialize DVC
dvc init
git commit -m 'Initialize DVC'

# Set DVC remote to S3
dvc remote add -d storage s3://sagemaker-${region}-${account}/DEMO-cxr-dvc
git commit .dvc/config -m "Configure DVC remote"

# Set DVC cache to S3
dvc remote add s3cache s3://sagemaker-${region}-${account}/DEMO-cxr-dvc/cache
dvc config cache.s3 s3cache
dvc config core.analytics false

git add .dvc/config
git commit -m 'Update DVC config'

git push --set-upstream origin main

## Part 2: Session and MLflow Setup
---

Configure the SageMaker AI session and create an MLflow tracking server for experiment management.

In [ ]:
import boto3
import json
from datetime import datetime
from pathlib import Path

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.image_uris import get_training_image_uri
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import (
    ProcessingInput, ProcessingS3Input,
)
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute
from sagemaker.core import image_uris

# Setup session
sess = Session()
role = get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()
account = boto3.client('sts').get_caller_identity()['Account']

dvc_repo_url = f"codecommit::{region}://{dvc_repo_name}"
prefix = 'DEMO-cxr-dvc'

print(f"Account: {account}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")
print(f"Role: {role}")

### Setup MLflow Tracking

Create or connect to a SageMaker AI MLflow App for experiment tracking.

> **Estimated time:** Creating a new MLflow App takes ~3-5 minutes.

**Note:** The following cells create an IAM role and MLflow App programmatically. Your notebook's IAM role must have `iam:CreateRole` and `iam:PutRolePolicy` permissions. 

Alternatively, you can create the MLflow App via the [Amazon SageMaker AI console](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-create-tracking-server-studio.html) and skip the role creation cell — just update `mlflow_app_name` to match your existing app.

In [ ]:
import mlflow
import time

sm_client = boto3.client('sagemaker')
iam = boto3.client('iam')

experiment_name = 'demo-cxr-mlflow-dvc'
mlflow_app_name = 'cxr-mlflow-app'
mlflow_role_name = 'MLflowAppIAMRole'

# Create IAM role for MLflow if needed
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

# Least-privilege policy for MLflow App role
# Based on AWS docs: https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-app-setup-prerequisites-iam.html
# S3 actions scoped to SageMaker bucket only
mlflow_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:Get*",
                "s3:Put*",
                "s3:List*"
            ],
            "Resource": [
                f"arn:aws:s3:::{bucket}",
                f"arn:aws:s3:::{bucket}/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:AddTags",
                "sagemaker:CreateModelPackageGroup",
                "sagemaker:CreateModelPackage",
                "sagemaker:UpdateModelPackage",
                "sagemaker:DescribeModelPackageGroup"
            ],
            "Resource": "*"
        }
    ]
}

try:
    iam.get_role(RoleName=mlflow_role_name)
    print(f"Using existing role: {mlflow_role_name}")
except iam.exceptions.NoSuchEntityException:
    print(f"Creating IAM role: {mlflow_role_name}")
    iam.create_role(
        RoleName=mlflow_role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='IAM role for MLflow App'
    )
    iam.put_role_policy(
        RoleName=mlflow_role_name,
        PolicyName='MLflowAppAccess',
        PolicyDocument=json.dumps(mlflow_policy)
    )
    # Wait for IAM role to propagate before using it
    print("Waiting for IAM role to propagate...")
    time.sleep(10)

mlflow_role_arn = f"arn:aws:iam::{account}:role/{mlflow_role_name}"
print(f"MLflow Role ARN: {mlflow_role_arn}")

In [ ]:
# Check for existing MLflow App by name, or create new one
apps = sm_client.list_mlflow_apps().get('Summaries', [])
mlflow_app = next((a for a in apps if a['Name'] == mlflow_app_name), None)

if mlflow_app:
    print(f"Using existing MLflow App: {mlflow_app['Name']}")
    mlflow_app = sm_client.describe_mlflow_app(Arn=mlflow_app['Arn'])
else:
    print(f"Creating MLflow App: {mlflow_app_name}...")
    response = sm_client.create_mlflow_app(
        Name=mlflow_app_name,
        ArtifactStoreUri=f's3://{bucket}',
        RoleArn=mlflow_role_arn
    )
    while True:
        mlflow_app = sm_client.describe_mlflow_app(Arn=response['Arn'])
        if mlflow_app['Status'] == 'Created':
            break
        elif mlflow_app['Status'] in ['CreateFailed', 'Deleted']:
            raise RuntimeError(f"MLflow App creation failed: {mlflow_app['Status']}")
        print(f"Status: {mlflow_app['Status']}... waiting")
        time.sleep(30)

mlflow_app_arn = mlflow_app['Arn']
print(f"MLflow App ARN: {mlflow_app_arn}")

## Part 3: Prepare Montgomery County CXR Dataset
---

Here we download the [Montgomery County Chest X-Ray Dataset](https://lhncbc.nlm.nih.gov/LHC-downloads/downloads.html#702702-tuberculosis-chest-x-ray-image-data-sets) from the National Library of Medicine (NLM), organize images by class (normal vs tuberculosis) based on clinical readings, upload to S3, and generate a patient consent manifest with randomly assigned patient IDs.

The dataset contains 138 posterior-anterior chest X-rays — 80 normal and 58 with tuberculosis manifestations — collected by the Department of Health and Human Services, Montgomery County, Maryland.

> **Note:** If the dataset has already been downloaded (the `MontgomerySet/` folder exists locally), the download step is skipped.

In [ ]:
from setup_cxr_dataset import setup_dataset

raw_data_s3_uri = setup_dataset(bucket, prefix)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, cls in zip(axes, ["normal", "tuberculosis"]):
    img_path = sorted((Path("MontgomerySet") / cls).glob("*.png"))[0]
    ax.imshow(Image.open(img_path), cmap="gray")
    ax.set_title(cls.capitalize())
    ax.axis("off")
plt.suptitle("Montgomery County CXR Dataset — Sample Images")
plt.tight_layout()
plt.show()

## Part 4: Process, Version, and Train (v1.0)
---

Upload the patient consent manifest to S3 and run the processing job. The processing job reads raw chest X-ray images from S3, filters for active patients based on the consent manifest, preprocesses images (resize to 128x128), splits into train/validation/test at the patient level, saves to ImageFolder format, and versions the output with DVC. Then we train a MobileNetV3-Small classifier on the versioned dataset and register the model in MLflow.

In [ ]:
import pandas as pd

# Define version for experiment 1
data_version_v1 = "v1.0"
timestamp = datetime.now().strftime("%m-%d-%y_%H%M")
pipeline_run_id_v1 = f"{data_version_v1}-{timestamp}"

# Load and inspect the patient consent manifest
registry = pd.read_csv("master_manifest.csv")
num_patients = registry['patient_id'].nunique()
num_active = registry.loc[registry['consent_status'] == 'active', 'patient_id'].nunique()
print(f"Patient manifest: {len(registry)} scans, {num_patients} patients, {num_active} active")
print(f"\nPipeline run ID: {pipeline_run_id_v1}")

# Upload manifest to S3
s3_client = boto3.client('s3')
registry_s3_prefix = f"{prefix}/registry"
registry_v1_s3_uri = f"s3://{bucket}/{registry_s3_prefix}/v1.0/"
s3_client.upload_file("master_manifest.csv", bucket, f"{registry_s3_prefix}/v1.0/manifest.csv")
print(f"Manifest uploaded to: {registry_v1_s3_uri}")

### Run Processing Job (v1.0)

> **Estimated time:** ~4-5 minutes

In [ ]:
# Get PyTorch image for processing
processing_image = get_training_image_uri(
    region=region,
    framework="pytorch",
    framework_version="2.6",
    py_version="py312",
    instance_type="ml.m5.xlarge",
)

processor_v1 = FrameworkProcessor(
    image_uri=processing_image,
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    env={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "PIPELINE_RUN_ID": pipeline_run_id_v1,
    }
)

print(f"Processing image: {processing_image}")

In [ ]:
%%time

processor_v1.run(
    code="preprocessing_healthcare.py",
    source_dir="../source_dir",
    inputs=[
        ProcessingInput(
            input_name="registry",
            s3_input=ProcessingS3Input(
                s3_uri=registry_v1_s3_uri,
                local_path="/opt/ml/processing/input/registry",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            )
        ),
        ProcessingInput(
            input_name="raw-data",
            s3_input=ProcessingS3Input(
                s3_uri=raw_data_s3_uri,
                local_path="/opt/ml/processing/input/raw-data/raw-cxr",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            )
        ),
    ],
    arguments=[
        "--data-version", data_version_v1,
        "--val-split", "0.15",
        "--test-split", "0.15",
    ],
    wait=True
)

### Train Model (v1.0)

> **Estimated time:** ~4-5 minutes. Includes instance provisioning, DVC pull of the versioned dataset, and training MobileNetV3-Small for 15 epochs.

In [ ]:
# Get PyTorch training image
training_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.6",
    py_version="py312",
    instance_type="ml.m5.xlarge",
    image_scope="training"
)

print(f"Training image: {training_image}")

In [ ]:
registered_model_name = "CXR-MobileNetV3"

model_trainer_v1 = ModelTrainer(
    sagemaker_session=sess,
    training_image=training_image,
    source_code=SourceCode(
        source_dir="../source_dir",
        entry_script="train.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type="ml.m5.xlarge",
        instance_count=1,
        volume_size_in_gb=30,
    ),
    base_job_name="cxr-mobilenet-v1",
    hyperparameters={
        "epochs": 15,
        "batch_size": 32,
        "learning_rate": 0.0001,
    },
    environment={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "DATA_VERSION": data_version_v1,
        "PIPELINE_RUN_ID": pipeline_run_id_v1,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "MLFLOW_REGISTERED_MODEL_NAME": registered_model_name,
    }
)

In [ ]:
%%time

model_trainer_v1.train()

## Part 5: Patient Opt-Out and Retrain (v2.0)
---

This section demonstrates how to handle a patient opting out of model training. The key insight: **the processing code never changes**. We simply update the patient consent manifest and run the same pipeline.

> **Estimated time:** ~8-10 minutes total for this section. The v2.0 processing job (~4-5 min) and training job (~4-5 min)

**Healthcare scenario:** A patient opts out of having their data used in model training. We must:
1. **Update their consent status** to `revoked` in the manifest
2. **Run the same processing job** with the updated manifest (same raw data, same code)
3. **DVC versions the new processed dataset** (v2.0) — automatically excludes their images
4. **Retrain** the model on the clean dataset
5. **Verify** via audit queries that the patient is not in the new model

> **Production considerations:** This demo uses a CSV file as the patient consent manifest. In production, consent status would live in a database (e.g., a consent management platform, patient registry, or purpose-built DynamoDB table) and the processing job would query it directly. When a patient revokes consent, their data is excluded from all future training jobs and experiments — the timing and cadence of retraining depends on your regulatory requirements.

In [ ]:
# Select a patient to opt out (pick one with 2+ scans for a visible demo)
registry = pd.read_csv("master_manifest.csv")
patient_scan_counts = registry.groupby('patient_id').size()
multi_scan_patients = patient_scan_counts[patient_scan_counts >= 2].index.tolist()
opt_out_patient = multi_scan_patients[5]  # Deterministic pick

num_patients = registry['patient_id'].nunique()
num_active = registry.loc[registry['consent_status'] == 'active', 'patient_id'].nunique()
print(f"Manifest: {len(registry)} scans, {num_patients} patients, {num_active} active")

patient_rows = registry[registry['patient_id'] == opt_out_patient]
print(f"\nOpt-out target: {opt_out_patient}")
print(f"  Scans: {len(patient_rows)}")
print(f"  Current status: {patient_rows['consent_status'].values[0]}")

In [ ]:
# Revoke consent for all of this patient's scans
# In production this would be a database UPDATE:
# UPDATE patient_consent SET status = 'revoked' WHERE patient_id = ?
registry.loc[registry['patient_id'] == opt_out_patient, 'consent_status'] = 'revoked'

# Verify
revoked_count = (registry['patient_id'] == opt_out_patient).sum()
active_patients = registry.loc[registry['consent_status'] == 'active', 'patient_id'].nunique()
print(f"{opt_out_patient}: revoked ({revoked_count} scans)")
print(f"Active patients: {active_patients}")

# Save updated manifest and upload to S3
registry_v2_path = "registry_v2.csv"
registry.to_csv(registry_v2_path, index=False)

registry_v2_s3_uri = f"s3://{bucket}/{registry_s3_prefix}/v2.0/"
s3_client.upload_file(registry_v2_path, bucket, f"{registry_s3_prefix}/v2.0/manifest.csv")
print(f"Updated manifest uploaded to: {registry_v2_s3_uri}")

In [ ]:
%%time

# Same processing code, updated manifest
data_version_v2 = "v2.0"
timestamp = datetime.now().strftime("%m-%d-%y_%H%M")
pipeline_run_id_v2 = f"{data_version_v2}-{timestamp}"

processor_v2 = FrameworkProcessor(
    image_uri=processing_image,
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    env={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "PIPELINE_RUN_ID": pipeline_run_id_v2,
    }
)

processor_v2.run(
    code="preprocessing_healthcare.py",
    source_dir="../source_dir",
    inputs=[
        ProcessingInput(
            input_name="registry",
            s3_input=ProcessingS3Input(
                s3_uri=registry_v2_s3_uri,
                local_path="/opt/ml/processing/input/registry",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            )
        ),
        ProcessingInput(
            input_name="raw-data",
            s3_input=ProcessingS3Input(
                s3_uri=raw_data_s3_uri,
                local_path="/opt/ml/processing/input/raw-data/raw-cxr",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            )
        ),
    ],
    arguments=[
        "--data-version", data_version_v2,
        "--val-split", "0.15",
        "--test-split", "0.15",
    ],
    wait=True
)

print(f"\nDataset v2.0 processed (without {opt_out_patient})")
print(f"Pipeline run ID: {pipeline_run_id_v2}")

In [ ]:
# Train v2.0 model on clean dataset (without opted-out patient)
model_trainer_v2 = ModelTrainer(
    sagemaker_session=sess,
    training_image=training_image,
    source_code=SourceCode(
        source_dir="../source_dir",
        entry_script="train.py",
        requirements="requirements.txt",
    ),
    compute=Compute(
        instance_type="ml.m5.xlarge",
        instance_count=1,
        volume_size_in_gb=30,
    ),
    base_job_name="cxr-mobilenet-v2",
    hyperparameters={
        "epochs": 15,
        "batch_size": 32,
        "learning_rate": 0.0001,
    },
    environment={
        "DVC_REPO_URL": dvc_repo_url,
        "DVC_REPO_NAME": dvc_repo_name,
        "DATA_VERSION": data_version_v2,
        "PIPELINE_RUN_ID": pipeline_run_id_v2,
        "MLFLOW_TRACKING_URI": mlflow_app_arn,
        "MLFLOW_EXPERIMENT_NAME": experiment_name,
        "MLFLOW_REGISTERED_MODEL_NAME": registered_model_name,
    }
)

print(f"Training v2.0 model (without {opt_out_patient})")

In [ ]:
%%time

model_trainer_v2.train()

## Part 6: Compare Experiments in MLflow
---

Now you can compare the two experiments in the MLflow UI. To access the UI, see [Launch the MLflow UI using a presigned URL](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-launch-ui.html).

You should see:
- **v1.0**: Trained with all patients (including the opted-out patient)
- **v2.0**: Trained without the opted-out patient

Each run includes:
- `data_version` and `data_git_commit_id` linking to the exact DVC dataset
- `patient_count` showing how many patients were in the training data
- `manifest.csv` artifact listing every patient and scan in the training set

![MLflow Experiment Comparison](../img/mlflow_experiment.png)

### Training Run Details
---

Click into any run to see detailed metrics, parameters, and artifacts. Key information includes:
- Training/validation loss curves over epochs
- Hyperparameters used (learning rate, batch size, epochs)
- DVC data version and Git commit linking the exact dataset

![MLflow Training Run Details](../img/mlflow_training_run.png)

### Registered Model
---

Models are automatically registered in the MLflow Model Registry. This provides:
- Version history of all trained models
- Stage transitions (Staging → Production)
- Direct links to the training run and data version that produced each model
- **Manifest artifacts** for patient-level audit on each model version

![MLflow Registered Model](../img/mlflow_registered_model.png)

## Part 7: Lineage Audit Queries
---

Now we answer audit questions by querying MLflow artifacts. Each query downloads a training run's `manifest.csv` artifact and checks for specific record IDs.

**Healthcare scenario — audit query:** "Which models were trained using this patient's scans?"

We run this for two patients to show the contrast:
- **Active patient** — should appear in both v1.0 and v2.0 models
- **Opted-out patient** — should appear only in v1.0 (excluded from v2.0 after consent revocation)

> This query generalizes to any record type: "Which models used customer X's data?", "Which models included transaction Y?", etc.

> **Production note:** The query below downloads the `manifest.csv` artifact from every training run and scans it — this works fine for a handful of runs but doesn't scale. In production, consider:
> - **Lineage index table** — At training time, write (record_id, run_id, data_version) tuples to DynamoDB or RDS. Audit queries become indexed lookups instead of artifact downloads.
> - **Athena over S3** — Point an Athena table at the MLflow artifact prefix in S3 and query manifests with SQL directly — no downloading or parsing in Python.
> - **Event-driven indexing** — A post-training Lambda reads the manifest and populates an index. Opt-out requests trigger immediate lookups and can auto-flag affected deployed endpoints for retraining.

In [ ]:
from utils.audit_queries import find_models_with_patient

# Connect to MLflow
mlflow.set_tracking_uri(mlflow_app_arn)

# Pick an active patient for comparison (one who stayed in both model versions)
active_patient = next(p for p in multi_scan_patients if p != opt_out_patient)

# Query: "Which models were trained on this patient's data?"
# Compare an active patient (should appear in both v1.0 and v2.0)
# vs. the opted-out patient (should appear only in v1.0)
for patient, label in [(active_patient, "active patient"), (opt_out_patient, "opted-out patient")]:
    print("=" * 60)
    print(f"AUDIT QUERY: Which models used {patient} ({label})?")
    print("=" * 60)
    find_models_with_patient(patient, experiment_name=experiment_name)
    print()

## Part 8: Deploy Model with ModelBuilder
---

Deploy the latest model (v2.0, trained on the clean dataset) from MLflow registry to a SageMaker AI endpoint.

> **Estimated time:** ~4-5 minutes for endpoint deployment.

In [ ]:
from mlflow import MlflowClient
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.serve.mode.function_pointers import Mode

# Connect to MLflow
mlflow.set_tracking_uri(mlflow_app_arn)
client = MlflowClient()

# Get the latest model version
registered_model = client.get_registered_model(name=registered_model_name)
latest_version = registered_model.latest_versions[0]

model_version = latest_version.version
model_source = latest_version.source
mlflow_model_path = f"models:/{registered_model_name}/{model_version}"

print(f"Model: {registered_model_name}")
print(f"Version: {model_version}")
print(f"Source: {model_source}")

In [ ]:
import torch
import json
import io
from PIL import Image
from torchvision import transforms
from sagemaker.serve.marshalling.custom_payload_translator import CustomPayloadTranslator

class ImageInputTranslator(CustomPayloadTranslator):
    """Handles raw image bytes (JPEG/PNG)"""
    def __init__(self):
        super().__init__(content_type='image/jpeg', accept_type='application/json')
        self.transform = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def serialize_payload_to_bytes(self, payload: object) -> bytes:
        # Client-side: send raw image bytes
        if isinstance(payload, bytes):
            return payload
        raise ValueError('Expected bytes')
    
    def deserialize_payload_from_stream(self, stream) -> object:
        # Server-side: decode image and transform to tensor
        image_bytes = stream.read()
        image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        tensor = self.transform(image).unsqueeze(0)
        # Move to GPU if available (matches model device)
        if torch.cuda.is_available():
            tensor = tensor.cuda()
        return tensor

class ImageOutputTranslator(CustomPayloadTranslator):
    """Converts model output tensor to JSON."""
    def __init__(self):
        super().__init__(content_type='application/json', accept_type='application/json')
    
    def serialize_payload_to_bytes(self, payload: object) -> bytes:
        if isinstance(payload, torch.Tensor):
            return json.dumps(payload.tolist()).encode('utf-8')
        return json.dumps(payload).encode('utf-8')
    
    def deserialize_payload_from_stream(self, stream) -> object:
        return json.load(stream)

# Sample input: raw JPEG bytes
sample_image = Image.new('RGB', (128, 128), color='gray')
buffer = io.BytesIO()
sample_image.save(buffer, format='JPEG')
sample_input = buffer.getvalue()

# Sample output: class probabilities for 2 classes (normal, tuberculosis)
sample_output = [[0.5] * 2]

schema_builder = SchemaBuilder(
    sample_input=sample_input,
    sample_output=sample_output,
    input_translator=ImageInputTranslator(),
    output_translator=ImageOutputTranslator()
)

In [ ]:
from sagemaker.core import image_uris

inference_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.6.0",
    py_version="py312",
    instance_type="ml.m5.xlarge",
    image_scope="inference"
)

In [ ]:
model_builder = ModelBuilder(
    mode=Mode.SAGEMAKER_ENDPOINT,
    image_uri=inference_image,
    instance_type="ml.m5.xlarge",
    schema_builder=schema_builder,
    model_metadata={
        "MLFLOW_MODEL_PATH": mlflow_model_path,
        "MLFLOW_TRACKING_ARN": mlflow_app_arn
    },
    dependencies={"auto": False, "custom": [
        "mlflow==3.4.0",
        "sagemaker-mlflow>=0.2.0",
        "sagemaker==3.4.0",
        "cloudpickle==3.1.2",
        "numpy==2.4.1"
    ]},
)

print(f"ModelBuilder configured with: {mlflow_model_path}")

In [ ]:
import uuid

unique_id = str(uuid.uuid4())[:8]
model_name = f"cxr-mobilenet-{unique_id}"
endpoint_name = f"cxr-endpoint-{unique_id}"

# Build the model
core_model = model_builder.build(model_name=model_name, region=region)
print(f"Model built: {core_model.model_name}")

In [ ]:
core_endpoint = model_builder.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1
)

print(f"Endpoint deployed: {core_endpoint.endpoint_name}")

### Test the Endpoint

In [ ]:
import json
import numpy as np
from pathlib import Path
from io import BytesIO
from PIL import Image

# Load a normal chest X-ray from the local Montgomery dataset for testing
normal_dir = Path("MontgomerySet/normal")
test_image_path = sorted(normal_dir.glob("*.png"))[0]

with open(test_image_path, 'rb') as f:
    image_bytes = f.read()

img = Image.open(BytesIO(image_bytes))
img.thumbnail((300, 300))
print(f"Test image: {test_image_path.name}")
img

In [ ]:
runtime_client = boto3.client('sagemaker-runtime')
response = runtime_client.invoke_endpoint(
    EndpointName=core_endpoint.endpoint_name,
    Body=image_bytes,
    ContentType='image/jpeg'
)
prediction = json.loads(response['Body'].read().decode('utf-8'))

predicted_class = np.argmax(prediction)
confidence = prediction[0][predicted_class]

class_names = ['normal', 'tuberculosis']
ground_truth = 'tuberculosis' if 'tuberculosis' in str(test_image_path) else 'normal'
correct = class_names[predicted_class] == ground_truth

print(f'Ground truth: {ground_truth}')
print(f'Predicted:    {class_names[predicted_class]} ({"correct" if correct else "incorrect"})')
print(f'\nAll probabilities:')
for name, prob in zip(class_names, prediction[0]):
    print(f'  {name}: {prob:.3f}')

## Cleanup
---

Delete resources to avoid ongoing charges.

In [ ]:
# Delete endpoint
from sagemaker.core.resources import EndpointConfig

core_endpoint_config = EndpointConfig.get(endpoint_config_name=core_endpoint.endpoint_name)
core_model.delete()
core_endpoint.delete()
core_endpoint_config.delete()

print("Endpoint resources deleted!")

In [ ]:
# Optional: Delete MLflow App
# sm_client.delete_mlflow_app(Arn=mlflow_app_arn)

In [ ]:
# Optional: Delete CodeCommit repository
# %aws codecommit delete-repository --repository-name {dvc_repo_name}

# Optional: Delete raw chest X-ray data from S3
# !aws s3 rm s3://{bucket}/{prefix}/raw-cxr --recursive

## Going Further
---

This demo shows the core pattern for record-level data lineage with DVC and MLflow on SageMaker AI using chest X-ray images. To move toward production-grade audit readiness consider:

- **Opt-out chain of custody** — Record when the opt-out request was received, when retraining completed, and when the clean model was deployed. The gap is the exposure window. Consider that derived artifacts (embeddings, cached features) generated before the opt-out also need invalidation
- **Deployment history** — Log which model version was serving each SageMaker AI endpoint and when (via CloudTrail or EventBridge). This demo tracks what data trained a model — deployment history closes the loop by proving whether that model was in production on a given date
- **Scale audit queries** — Replace the manifest-download approach in Part 5 with an index (DynamoDB, Athena, or a post-training Lambda) for sub-second lookups across thousands of models
- **Flag affected endpoints** — When a record is excluded, automatically identify deployed models trained on that record and flag them for retraining
- **Tamper-proof manifests** — Store manifests in S3 with [Object Lock](https://docs.aws.amazon.com/AmazonS3/latest/userguide/object-lock.html) and write a SHA-256 hash of each manifest to an independent, append-only store (e.g., a separate Object Lock bucket or an audit ledger). At audit time, re-hash the manifest and verify against the independent record — proves the manifest wasn't modified after training
- **Reproducibility** — Given any model in MLflow, extract its `data_git_commit_id`, run `git checkout <tag> && dvc pull` to recreate the exact training data

### Speeding Up Iteration

When moving from a demo like this to a production compliance workflow where retraining happens frequently (e.g., after each opt-out), two SageMaker AI features help streamline the process:

- **[SageMaker AI Managed Warm Pools](https://docs.aws.amazon.com/sagemaker/latest/dg/train-warm-pools.html)** — Keep training instances warm between jobs so back-to-back training runs (like v1.0 → v2.0 above) reuse already-provisioned infrastructure. Add `keep_alive_period_in_seconds` to your `Compute` config to enable it. Note that warm pools apply to training jobs only, not processing jobs.

- **[SageMaker AI Pipelines](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines-overview.html)** — Orchestrate the processing → training → registration workflow as a single, repeatable pipeline instead of running each step manually in a notebook. Pipelines handle step dependencies, pass artifacts between steps automatically, and can be triggered programmatically (e.g., when a patient opts out and the manifest is updated). This eliminates the repeated boilerplate you see in Parts 4 and 5, where the same processor/trainer configuration is defined twice with only the manifest version changing.